In [1]:
import math
import random
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import community as community_louvain
from collections import Counter
 
random.seed(42)
np.random.seed(42)

In [2]:
print("\n[1/6] Importazione grafo")
G = nx.read_edgelist("Dataset/facebook_combined.txt")
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
tutti_i_gradi = [grado for nodo, grado in G.degree()]
grado_massimo = max(tutti_i_gradi)
grado_minimo = min(tutti_i_gradi)
print(f"      Nodi: {n_nodes}, Archi: {n_edges}")
print(f"      Grado medio: {2*n_edges/n_nodes:.2f}")
print(f"      Grado massimo: {grado_massimo}")
print(f"      Grado minimo: {grado_minimo}")
print(f"      Clustering medio: {nx.average_clustering(G):.4f}")
out_path = "/Users/giuseppepiosorrentino/RetiSocialiProj/Results/MIO/"



[1/6] Importazione grafo
      Nodi: 4039, Archi: 88234
      Grado medio: 43.69
      Grado massimo: 1045
      Grado minimo: 1
      Clustering medio: 0.6055


In [3]:
def precompute_graph(G):
    neighbors  = {v: set(G.neighbors(v)) for v in G.nodes()}
    degrees    = {v: G.degree(v) for v in G.nodes()}
    thresholds = {v: math.ceil(degrees[v] / 2) for v in G.nodes()}
    return neighbors, degrees, thresholds
 
neighbors, degrees, thresholds = precompute_graph(G)
 

In [4]:
import random
import math

def cost_random(G, low=1, high=5, seed=42):
    rng = random.Random(seed)
    return {v: rng.randint(low, high) for v in G.nodes()}

def cost_half_degree(G):
    return {v: max(1, math.ceil(G.degree(v) / 2)) for v in G.nodes()}

#VALUTIAMO, è OPZIONALE
def cost_custom(G):
    clust = nx.clustering(G)
    return {v: max(1, round(1 + clust[v] * 10)) for v in G.nodes()}

# Istanzia le tre funzioni costo sul grafo G
c_random = cost_random(G)
c_half   = cost_half_degree(G)
#OPZIONALE
c_custom = cost_custom(G)

print("Esempio costi (primi 5 nodi):")
for v in list(G.nodes())[:5]:
    print(f"  nodo {v}: random={c_random[v]}, d/2={c_half[v]}, custom={c_custom[v]}")



Esempio costi (primi 5 nodi):
  nodo 0: random=1, d/2=174, custom=1
  nodo 1: random=1, d/2=9, custom=5
  nodo 2: random=3, d/2=5, custom=10
  nodo 3: random=2, d/2=9, custom=7
  nodo 4: random=2, d/2=5, custom=10


In [5]:
costo_totale_random = sum(c_random.values())
costo_totale_half   = sum(c_half.values())
costo_totale_custom = sum(c_custom.values())
k_random = int(0.10 * costo_totale_random)
k_half   = int(0.10 * costo_totale_half)
k_custom = int(0.10 * costo_totale_custom)

print(f"Costo totale random: {costo_totale_random}")
print(f"Costo totale half:   {costo_totale_half}")
print(f"Costo totale custom: {costo_totale_custom}")

Costo totale random: 12173
Costo totale half:   89243
Costo totale custom: 28467


In [6]:
def majority_cascade(G, S):
    infected = set(S)
    while True:
        new_infected = set()
        for v in G.nodes():
            if v not in infected:
                dv = G.degree(v)
                if dv == 0:
                    continue
                if sum(1 for u in G.neighbors(v) if u in infected) >= math.ceil(dv / 2):
                    new_infected.add(v)
        if not new_infected:
            break
        infected |= new_infected
    return infected

In [7]:
print("Community detection con Louvain...")
partition = community_louvain.best_partition(G, random_state=42)
 
n_communities = len(set(partition.values()))
sizes = Counter(partition.values())
print(f"Comunità trovate:    {n_communities}")
print(f"Dimensione media:    {sum(sizes.values())/len(sizes):.1f}")
print(f"Comunità più grande: {max(sizes.values())} nodi")
print(f"Comunità più piccola:{min(sizes.values())} nodi")
 

Community detection con Louvain...
Comunità trovate:    15
Dimensione media:    269.3
Comunità più grande: 548 nodi
Comunità più piccola:19 nodi


In [8]:
print("Calcolo betweenness centrality (potrebbe richiedere qualche minuto)...")
bc = nx.betweenness_centrality(G, normalized=True)
 
def external_edges(G, partition):
    """Numero di archi di v verso nodi in comunità diverse."""
    return {v: sum(1 for u in G.neighbors(v) if partition[u] != partition[v])
            for v in G.nodes()}
 
ext = external_edges(G, partition)
 
# Normalizza in [0,1]
bc_vals  = np.array([bc[v]       for v in G.nodes()])
deg_vals = np.array([degrees[v]  for v in G.nodes()])
ext_vals = np.array([ext[v]      for v in G.nodes()])
 
bc_norm  = bc_vals  / bc_vals.max()
deg_norm = deg_vals / deg_vals.max()
ext_norm = ext_vals / ext_vals.max() if ext_vals.max() > 0 else ext_vals
 
bc_norm_dict  = {v: bc_norm[i]  for i, v in enumerate(G.nodes())}
deg_norm_dict = {v: deg_norm[i] for i, v in enumerate(G.nodes())}
ext_norm_dict = {v: ext_norm[i] for i, v in enumerate(G.nodes())}

Calcolo betweenness centrality (potrebbe richiedere qualche minuto)...


In [9]:
def cacmc(G, k, c, partition, bc_norm, deg_norm, ext_norm,
          alpha=0.4, beta=0.3, gamma=0.3):
    """
    Community-Aware Cost Majority Cascade (CACMC).
 
    Intuizione: in reti ad alto clustering come Facebook, una volta
    penetrata una comunità la propagazione interna è quasi automatica.
    Il problema vero non è attivare tanti nodi localmente, ma
    innescare comunità differenti.
 
    Score per ogni nodo v:
      score(v) = (α·BC(v) + β·Deg(v) + γ·Ext(v)) / c(v) * penalità_comunità
 
    dove:
      BC(v)  = betweenness centrality  → nodi ponte tra comunità
      Deg(v) = grado normalizzato      → capacità di diffusione locale
      Ext(v) = archi verso altre comunità → diversità strutturale
    """
    S = set()
    current_cost = 0
    community_coverage = Counter()  # seed già scelti per comunità
 
    while True:
        best_u     = None
        best_score = -1
 
        for v in set(G.nodes()) - S:
            if current_cost + c[v] > k:
                continue
            if degrees[v] == 0:
                continue
 
            structural_score = (alpha * bc_norm[v] +
                                beta  * deg_norm[v] +
                                gamma * ext_norm[v])
 
            # Penalità: meno valore se la comunità è già coperta
            coverage_penalty = 1 / (1 + community_coverage[partition[v]])
 
            score = (structural_score * coverage_penalty) / c[v]
 
            if score > best_score:
                best_score = score
                best_u = v
 
        if best_u is None:
            break
 
        S.add(best_u)
        current_cost += c[best_u]
        community_coverage[partition[best_u]] += 1
 
    return S
 

In [10]:
def remove_edges(G, x, seed=42):
    G2 = G.copy()
    rng = random.Random(seed)
    to_remove = rng.sample(list(G2.edges()), min(x, G2.number_of_edges()))
    G2.remove_edges_from(to_remove)
    return G2
 
def remove_vertices(G, y, seed=42):
    G2 = G.copy()
    rng = random.Random(seed)
    to_remove = rng.sample(list(G2.nodes()), min(y, G2.number_of_nodes()))
    G2.remove_nodes_from(to_remove)
    return G2

In [11]:
percentages = [0.05, 0.06, 0.07, 0.08, 0.09,
               0.10, 0.12, 0.15, 0.20, 0.25,
               0.30, 0.40, 0.50]
 
total_random = sum(c_random.values())
total_half   = sum(c_half.values())
total_custom = sum(c_custom.values())
 
budgets = {
    "c_random": [int(p * total_random) for p in percentages],
    "c_half":   [int(p * total_half)   for p in percentages],
    "c_custom": [int(p * total_custom) for p in percentages],
}
 
cost_funcs = {
    "c_random": c_random,
    "c_half":   c_half,
    "c_custom": c_custom,
}

In [12]:
print("\n=== STEP 1: |Inf[G,S]| al variare di k ===")
 
inf_results = {}  # inf_results[cost_name] = lista |Inf|
seeds_ref   = {}  # seeds_ref[cost_name] = seed set al k di riferimento (10%)
 
for cost_name, c in cost_funcs.items():
    print(f"\n  Funzione costo: {cost_name}")
    inf_list = []
    k_ref = int(0.10 * sum(c.values()))
 
    for k in budgets[cost_name]:
        S   = cacmc(G, k, c, partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
        inf = majority_cascade(G, S)
        inf_list.append(len(inf))
 
        n_comm_covered = len(set(partition[v] for v in S))
        print(f"    k={k:6d} | seed={len(S):4d} | |Inf|={len(inf):4d} | "
              f"comunità coperte={n_comm_covered}/{n_communities}")
 
        if k == k_ref:
            seeds_ref[cost_name] = S
 
    inf_results[cost_name] = inf_list
 


=== STEP 1: |Inf[G,S]| al variare di k ===

  Funzione costo: c_random
    k=   608 | seed= 449 | |Inf|= 852 | comunità coperte=15/15
    k=   730 | seed= 516 | |Inf|= 985 | comunità coperte=15/15
    k=   852 | seed= 577 | |Inf|=1439 | comunità coperte=15/15
    k=   973 | seed= 638 | |Inf|=2104 | comunità coperte=15/15
    k=  1095 | seed= 691 | |Inf|=2147 | comunità coperte=15/15
    k=  1217 | seed= 741 | |Inf|=2777 | comunità coperte=15/15
    k=  1460 | seed= 843 | |Inf|=3319 | comunità coperte=15/15
    k=  1825 | seed= 984 | |Inf|=3357 | comunità coperte=15/15
    k=  2434 | seed=1192 | |Inf|=3418 | comunità coperte=15/15
    k=  3043 | seed=1377 | |Inf|=3445 | comunità coperte=15/15
    k=  3651 | seed=1572 | |Inf|=3586 | comunità coperte=15/15
    k=  4869 | seed=1963 | |Inf|=3763 | comunità coperte=15/15
    k=  6086 | seed=2337 | |Inf|=3891 | comunità coperte=15/15

  Funzione costo: c_half
    k=  4462 | seed= 165 | |Inf|= 395 | comunità coperte=15/15
    k=  5354 | seed=

In [13]:
COLORS = {"c_random": "#2196F3", "c_half": "#E91E63", "c_custom": "#4CAF50"}
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
 
for ax, (cost_name, inf_list) in zip(axes, inf_results.items()):
    ax.plot([p*100 for p in percentages], inf_list,
            color=COLORS[cost_name], marker="o", linewidth=2, label=cost_name)
    ax.axhline(y=G.number_of_nodes(), color="gray",
               linestyle="--", alpha=0.5, label=f"|V|={G.number_of_nodes()}")
    ax.set_title(f"CACMC — {cost_name}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Budget (% costo totale)", fontsize=10)
    ax.set_ylabel("|Inf[G, S]|", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
 
fig.suptitle("Step 1 — CACMC: |Inf[G,S]| al variare del budget k",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("cacmc_step1.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSalvato: cacmc_step1.png")


Salvato: cacmc_step1.png


In [15]:
# Seed set di riferimento calcolati con k=1217
k_random = 3651
k_half = 35697
k_custom = 8540
# Ripristina le strutture usate da wtss_maximal se "degrees" è stato sovrascritto (es. lista)
if not isinstance(degrees, dict):
    neighbors, degrees, thresholds = precompute_graph(G)

S_ref_c1 = cacmc(G, k_random, c_random,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
S_ref_c2 = cacmc(G, k_half, c_half,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
S_ref_c3 = cacmc(G, k_custom, c_custom,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
# Range di archi da rimuovere
x_range = list(range(0, 20000, 1000))

inf_c1_edges   = []
inf_c2_edges   = []
inf_c3_edges   = []


for x in x_range:
    G2 = remove_edges(G, x)
    inf_c1_edges.append(len(majority_cascade(G2, S_ref_c1)))
    inf_c2_edges.append(len(majority_cascade(G2, S_ref_c2)))
    inf_c3_edges.append(len(majority_cascade(G2, S_ref_c3)))
    print(f"x={x:5d} | c1={inf_c1_edges[-1]:4d} | c2={inf_c2_edges[-1]:4d} | c3={inf_c3_edges[-1]:4d} |")

x=    0 | c1=3586 | c2=2950 | c3=3466 |
x= 1000 | c1=3587 | c2=2970 | c3=3467 |
x= 2000 | c1=3587 | c2=2955 | c3=3482 |
x= 3000 | c1=3578 | c2=2957 | c3=3484 |
x= 4000 | c1=3577 | c2=2960 | c3=3472 |
x= 5000 | c1=3586 | c2=2970 | c3=3471 |
x= 6000 | c1=3605 | c2=2971 | c3=3469 |
x= 7000 | c1=3609 | c2=2972 | c3=3490 |
x= 8000 | c1=3606 | c2=2969 | c3=3489 |
x= 9000 | c1=3597 | c2=2972 | c3=3493 |
x=10000 | c1=3609 | c2=2971 | c3=3493 |
x=11000 | c1=3635 | c2=2972 | c3=3493 |
x=12000 | c1=3598 | c2=2980 | c3=3491 |
x=13000 | c1=3584 | c2=3068 | c3=3475 |
x=14000 | c1=3583 | c2=2974 | c3=3475 |
x=15000 | c1=3581 | c2=2974 | c3=3453 |
x=16000 | c1=3581 | c2=2972 | c3=3438 |
x=17000 | c1=3577 | c2=2974 | c3=3435 |
x=18000 | c1=3631 | c2=2974 | c3=3473 |
x=19000 | c1=3630 | c2=3067 | c3=3458 |


In [18]:
plt.figure(figsize=(9, 5))

plt.plot(x_range, inf_c1_edges,   color="#E91E63", marker="s", linewidth=2, label="CACMC-Crandom")
plt.plot(x_range, inf_c2_edges,   color="#FF9800", marker="D", linewidth=2, label="CACMC-Chalf")
plt.plot(x_range, inf_c3_edges,   color="#9C27B0", marker="^", linewidth=2, label="CACMC-Ccustom")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
plt.axhline(y=3586, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=2950, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3466, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Archi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione archi ")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "edge_removalCACMC.png", dpi=150, bbox_inches="tight")


In [19]:
# Seed set di riferimento calcolati con k=1217
k_random = 3651
k_half = 35697
k_custom = 8540
# Ripristina le strutture usate da wtss_maximal se "degrees" è stato sovrascritto (es. lista)
if not isinstance(degrees, dict):
    neighbors, degrees, thresholds = precompute_graph(G)

S_ref_c1 = cacmc(G, k_random, c_random,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
S_ref_c2 = cacmc(G, k_half, c_half,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
S_ref_c3 = cacmc(G, k_custom, c_custom,partition,
                   bc_norm_dict, deg_norm_dict, ext_norm_dict)
# Range di archi da rimuovere
y_range = list(range(0, 1000, 50))

inf_c1_edges   = []
inf_c2_edges   = []
inf_c3_edges   = []


for y in y_range:
    G2 = remove_vertices(G, y)
    inf_c1_edges.append(len(majority_cascade(G2, S_ref_c1)))
    inf_c2_edges.append(len(majority_cascade(G2, S_ref_c2)))
    inf_c3_edges.append(len(majority_cascade(G2, S_ref_c3)))
    print(f"y={y:5d} | c1={inf_c1_edges[-1]:4d} | c2={inf_c2_edges[-1]:4d} | c3={inf_c3_edges[-1]:4d} |")

y=    0 | c1=3586 | c2=2950 | c3=3466 |
y=   50 | c1=3555 | c2=2937 | c3=3434 |
y=  100 | c1=3529 | c2=2924 | c3=3423 |
y=  150 | c1=3500 | c2=2903 | c3=3401 |
y=  200 | c1=3468 | c2=2893 | c3=3376 |
y=  250 | c1=3440 | c2=2872 | c3=3336 |
y=  300 | c1=3445 | c2=2857 | c3=3297 |
y=  350 | c1=3404 | c2=2842 | c3=3258 |
y=  400 | c1=3361 | c2=2818 | c3=3228 |
y=  450 | c1=3327 | c2=2880 | c3=3204 |
y=  500 | c1=3307 | c2=2866 | c3=3179 |
y=  550 | c1=3293 | c2=2765 | c3=3155 |
y=  600 | c1=3273 | c2=2805 | c3=3141 |
y=  650 | c1=3264 | c2=2876 | c3=3142 |
y=  700 | c1=3247 | c2=2785 | c3=3124 |
y=  750 | c1=3228 | c2=2787 | c3=3094 |
y=  800 | c1=3186 | c2=2798 | c3=3039 |
y=  850 | c1=3158 | c2=2768 | c3=3031 |
y=  900 | c1=3133 | c2=2751 | c3=3014 |
y=  950 | c1=3155 | c2=2710 | c3=2979 |


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(y_range, inf_c1_edges,   color="#E91E63", marker="s", linewidth=2, label="CACMC-Crandom")
plt.plot(y_range, inf_c2_edges,   color="#FF9800", marker="D", linewidth=2, label="CACMC-Chalf")
plt.plot(y_range, inf_c3_edges,   color="#9C27B0", marker="^", linewidth=2, label="CACMC-Ccustom")

# Linea di riferimento: |Inf| sul grafo originale senza rimozioni
plt.axhline(y=3586, color="#E91E63", linestyle="--", alpha=0.3)
plt.axhline(y=2950, color="#FF9800", linestyle="--", alpha=0.3)
plt.axhline(y=3466, color="#9C27B0", linestyle="--", alpha=0.3)

plt.xlabel("Nodi rimossi (x)")
plt.ylabel("|Inf[G', S]|")
plt.title("Step 2 — Effetto rimozione nodi ")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_path + "node_removalCACMC.png", dpi=150, bbox_inches="tight")



In [ ]:
print("CIAO")